In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Write |predicted_error - true_error| (|pred - oof_resid|) into the B-factor column of each
protein's predicted structure, for color-mapping the *prediction error's own error* in
PyMOL/ChimeraX.

Residues with no valid oof_resid (masked out, or beyond the deposited structure's length) get
the sentinel B-factor -1.00.
"""
import json
import os

import numpy as np

PRED_PATH = ("../../../data/analysis/Fig5/predictions_N_ensemble_10seed_sorted_centered_err.json")
STRUCT_DIR = "../../../data/rBMRB/cluster_30p/structures_split_filtered/test"
OUT_DIR = os.getcwd()

SENTINEL = -1.00

PIDS = ["17836_1", "30133_1", "5506_1"]


def diff_by_residue(pid, pred_json):
    """{orig_idx (0-based sequence position) -> |pred - oof_resid|} for one protein,
    same mask & oof_mask & finite filtering used throughout the oof-error analysis."""
    p = np.array(pred_json["pred"][pid], dtype=np.float64)
    mask = np.array(pred_json["mask"][pid], dtype=bool)
    oof_resid = np.array(pred_json["oof_resid"][pid], dtype=np.float64)
    oof_mask = np.array(pred_json["oof_mask"][pid], dtype=bool)
    n = min(len(p), len(mask), len(oof_resid), len(oof_mask))
    valid = mask[:n] & oof_mask[:n]
    pred_v = p[:n][valid]
    true_v = oof_resid[:n][valid]
    diff = np.abs(pred_v - true_v)
    finite = np.isfinite(diff)
    orig_idx = np.where(valid)[0][finite]
    diff = diff[finite]
    return dict(zip(orig_idx.tolist(), diff.tolist()))


def write_bfactor_pdb(pid, val_by_pos, out_path):
    struct_path = os.path.join(STRUCT_DIR, f"{pid}.pdb")
    with open(struct_path) as fh:
        lines = fh.readlines()

    out_lines = []
    n_written, n_sentinel = 0, 0
    for line in lines:
        if line.startswith(("ATOM", "HETATM")):
            resnum = int(line[22:26])
            orig_idx = resnum - 1
            val = val_by_pos.get(orig_idx, SENTINEL)
            if val == SENTINEL:
                n_sentinel += 1
            else:
                n_written += 1
            line = line[:60] + f"{val:6.2f}" + line[66:]
        out_lines.append(line)

    with open(out_path, "w") as fh:
        fh.writelines(out_lines)
    return n_written, n_sentinel


def main():
    pred_json = json.load(open(PRED_PATH))
    for pid in PIDS:
        val_by_pos = diff_by_residue(pid, pred_json)
        out_path = os.path.join(OUT_DIR, f"{pid}_pred_vs_true_diff_bfactor.pdb")
        n_written, n_sentinel = write_bfactor_pdb(pid, val_by_pos, out_path)
        worst = sorted(val_by_pos.items(), key=lambda kv: -kv[1])[:10]
        print(f"{pid}: wrote {out_path}")
        print(f"  {n_written} residues with real diff, {n_sentinel} sentinel (no data)")
        print(f"  worst 10 (1-based resi, |pred-true|): "
              f"{[(k + 1, round(v, 2)) for k, v in worst]}")


if __name__ == "__main__":
    main()


In [ ]:
main()